In [1]:
import pytesseract
pytesseract.pytesseract.tesseract_cmd = r'C:\Users\Alina\AppData\Local\Programs\Tesseract-OCR\tesseract.exe'

import fitz
from pdf2image import convert_from_path
from PIL import Image
import re
from pathlib import Path
from datetime import datetime
from difflib import SequenceMatcher
import json
import time

print("✓ All imports successful")
print(f"✓ Tesseract: {pytesseract.get_tesseract_version()}")
print(f"✓ Languages: {pytesseract.get_languages()}")

✓ All imports successful
✓ Tesseract: 5.5.0.20241111
✓ Languages: ['eng', 'fra', 'osd']


In [2]:
class MedicalProcessor:
    def __init__(self, output_dir="./pipeline_output", min_similarity=0.75):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        self.min_similarity = min_similarity
        self.patient_mapping = {}
        self.patient_counter = 1
        self.processed_folders = set()
        self.folder_patient_cache = {}
        self.load_mapping()
    
    def load_mapping(self):
        """Load existing mapping and track processed folders."""
        mapping_file = self.output_dir / "patient_mapping.json"
        if mapping_file.exists():
            with open(mapping_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                for item in data.get('mappings', []):
                    key = f"{item['lastname']}, {item['firstname']}"
                    self.patient_mapping[key] = {
                        'code': item['code'],
                        'lastname': item['lastname'],
                        'firstname': item['firstname']
                    }
                    num = int(item['code'].split('_')[1])
                    if num >= self.patient_counter:
                        self.patient_counter = num + 1
                
                if 'processed_folders' in data:
                    self.processed_folders = set(data['processed_folders'])
            
            print(f"✓ Loaded {len(self.patient_mapping)} existing patient(s)")
            if self.processed_folders:
                print(f"✓ Already processed: {len(self.processed_folders)} folder(s)")
    
    def ocr_pdf(self, pdf_path):
        """Extract text from PDF using PyMuPDF only."""
        try:
            doc = fitz.open(pdf_path)
            text = "".join(page.get_text() for page in doc)
            doc.close()
            if len(text.strip()) > 100:
                return text, "direct"
        except:
            pass
        
        try:
            doc = fitz.open(pdf_path)
            text = ""
            
            for page_num in range(len(doc)):
                page = doc[page_num]
                pix = page.get_pixmap(dpi=300)
                img_data = pix.tobytes("png")
                
                from io import BytesIO
                img = Image.open(BytesIO(img_data))
                page_text = pytesseract.image_to_string(img, lang='fra', config='--psm 1')
                text += f"\n--- Page {page_num+1} ---\n{page_text}"
            
            doc.close()
            return text, "ocr"
        except Exception as e:
            print(f"OCR error: {e}")
            raise
    
    def extract_patient_name(self, text):
        """
        Extract patient name - handles multi-word names.
        Uses negative lookahead to stop at correct boundary.
        """
        # Pattern 1: "de l'enfant LASTNAME, FIRSTNAME"
        pattern1 = r"de l['''ʼ]enfant\s+([A-ZÉÈÊËÀÂÄÔÖÛÜÇ][A-ZÉÈÊËÀÂÄÔÖÛÜÇ\s-]+?)(?:,\s*)([A-ZÉÈÊËÀÂÄÔÖÛÜÇ][A-ZÉÈÊËÀÂÄÔÖÛÜÇ\s-]+?)(?=\s*(?:\n|Né|né|NÉ|NÉE|Date|date|\d))"
        
        match = re.search(pattern1, text, re.IGNORECASE)
        if match:
            lastname = match.group(1).strip()
            firstname = match.group(2).strip()
            lastname = re.sub(r'[,;\.]$', '', lastname).strip()
            firstname = re.sub(r'[,;\.]$', '', firstname).strip()
            return lastname, firstname, "de l'enfant"
        
        # Pattern 2: "Nom: LASTNAME Prénom: FIRSTNAME"
        pattern2 = r"Nom\s*:\s*([A-ZÉÈÊËÀÂÄÔÖÛÜÇ][A-Za-zéèêëàâäôöûüç\s-]+?)(?:\s+Prénom|\s+Date)"
        pattern2b = r"Prénom\s*:\s*([A-ZÉÈÊËÀÂÄÔÖÛÜÇ][A-Za-zéèêëàâäôöûüç\s-]+?)(?:\s*\n|\s+Date|\s+Né)"
        
        match_lastname = re.search(pattern2, text, re.IGNORECASE)
        match_firstname = re.search(pattern2b, text, re.IGNORECASE)
        
        if match_lastname and match_firstname:
            lastname = match_lastname.group(1).strip().upper()
            firstname = match_firstname.group(1).strip().upper()
            return lastname, firstname, "Nom/Prénom"
        
        return None, None, None
    
    def similarity(self, s1, s2):
        return SequenceMatcher(None, s1.lower(), s2.lower()).ratio()
    
    def is_similar(self, snippet, lastname, firstname):
        """Check similarity to ANY component of patient name."""
        snippet_lower = snippet.lower()
        
        sims = [
            self.similarity(snippet_lower, lastname.lower()),
            self.similarity(snippet_lower, firstname.lower()),
            self.similarity(snippet_lower, f"{lastname} {firstname}".lower())
        ]
        
        for word in lastname.split():
            if len(word) > 2:
                sims.append(self.similarity(snippet_lower, word.lower()))
        
        for word in firstname.split():
            if len(word) > 2:
                sims.append(self.similarity(snippet_lower, word.lower()))
        
        return max(sims) >= self.min_similarity
    
    def get_code(self, lastname, firstname):
        key = f"{lastname.upper()}, {firstname.upper()}"
        if key not in self.patient_mapping:
            code = f"PATIENT_{self.patient_counter:03d}"
            self.patient_mapping[key] = {
                'code': code,
                'lastname': lastname,
                'firstname': firstname
            }
            self.patient_counter += 1
            return code
        return self.patient_mapping[key]['code']
    
    def remove_pii_structured(self, text):
        """Remove structured PII (NPI, NDA, addresses)."""
        anonymized = text
        
        # NPI numbers
        anonymized = re.sub(r'NPI\s*:?\s*\d+', 'NPI: [REMOVED]', anonymized, flags=re.IGNORECASE)
        
        # NDA numbers
        anonymized = re.sub(r'NDA\s*:?\s*\d+', 'NDA: [REMOVED]', anonymized, flags=re.IGNORECASE)
        
        # N° de séjour
        anonymized = re.sub(r'N°\s*de séjour\s*:?\s*\d+', 'N° de séjour: [REMOVED]', anonymized, flags=re.IGNORECASE)
        
        # Street addresses
        anonymized = re.sub(
            r'\d+\s+(?:ALLEE|AVENUE|RUE|BOULEVARD|PLACE|IMPASSE|CHEMIN|ROUTE|QUAI)\s+[A-ZÉÈÊËÀÂÄÔÖÛÜÇ\s]+(?:\n|$)',
            '[ADDRESS_REMOVED]\n',
            anonymized,
            flags=re.IGNORECASE
        )
        
        # Postal codes + city
        anonymized = re.sub(r'\d{5}\s+[A-ZÉÈÊËÀÂÄÔÖÛÜÇ\s]+(?:\n|$)', '[POSTAL_REMOVED]\n', anonymized, flags=re.IGNORECASE)
        
        # Address block
        anonymized = re.sub(
            r'Adresse\s+des\s+parents\s*:.*?(?=\n\s*\n|\n\s*[A-Z][a-z]|\Z)',
            'Adresse des parents: [ADDRESS_REMOVED]',
            anonymized,
            flags=re.IGNORECASE | re.DOTALL
        )
        
        # Code FINESS
        anonymized = re.sub(r'(?:Code\s+FINESS|FINESSE)\s*:?\s*\d+', 'Code FINESS: [REMOVED]', anonymized, flags=re.IGNORECASE)
        
        return anonymized
    
    def anonymize(self, text, lastname, firstname):
        """
        Complete anonymization with GUARANTEED full name removal.
        
        Strategy:
        1. Replace multi-word combinations FIRST (no word boundaries)
        2. Replace single words LAST (with word boundaries)
        3. This ensures "NDOUNKE DJATCHEU" is replaced before "NDOUNKE"
        """
        code = self.get_code(lastname, firstname)
        anonymized = text
        count = 0
        
        # STEP 1: Build variations in TWO categories
        multiword_variations = []
        singleword_variations = []
        
        # Full name combinations
        full_combos = [
            f"{lastname}, {firstname}",
            f"{lastname},{firstname}",
            f"{lastname} {firstname}",
            f"{firstname} {lastname}",
            lastname,
            firstname
        ]
        
        for combo in full_combos:
            if ' ' in combo or ',' in combo:
                multiword_variations.append(combo)
            else:
                singleword_variations.append(combo)
        
        # Individual words from multi-word names
        for word in lastname.split():
            if len(word) > 2:
                singleword_variations.append(word)
        
        for word in firstname.split():
            if len(word) > 2:
                singleword_variations.append(word)
        
        # STEP 2: Replace multi-word patterns FIRST (NO word boundaries)
        multiword_variations = sorted(set(multiword_variations), key=len, reverse=True)
        
        for var in multiword_variations:
            if not var.strip():
                continue
            
            pattern = re.escape(var)
            matches = len(re.findall(pattern, anonymized, re.IGNORECASE))
            if matches > 0:
                anonymized = re.sub(pattern, code, anonymized, flags=re.IGNORECASE)
                count += matches
        
        # STEP 3: Replace single-word patterns LAST (WITH word boundaries)
        singleword_variations = sorted(set(singleword_variations), key=len, reverse=True)
        
        for var in singleword_variations:
            if not var.strip():
                continue
            
            pattern = r'\b' + re.escape(var) + r'\b'
            matches = len(re.findall(pattern, anonymized, re.IGNORECASE))
            if matches > 0:
                anonymized = re.sub(pattern, code, anonymized, flags=re.IGNORECASE)
                count += matches
        
        # STEP 4: Fuzzy matching for OCR errors
        potential_errors = set(re.findall(r'\b[A-ZÉÈÊËÀÂÄÔÖÛÜÇ][A-ZÉÈÊËÀÂÄÔÖÛÜÇ\s-]{3,}\b', anonymized))
        for candidate in potential_errors:
            if code in candidate:
                continue
            if self.is_similar(candidate, lastname, firstname):
                anonymized = re.sub(re.escape(candidate), code, anonymized, flags=re.IGNORECASE)
                count += 1
        
        # STEP 5: Remove structured PII
        anonymized = self.remove_pii_structured(anonymized)
        
        # STEP 6: Remove doctor/staff names
        doctor_names = set(re.findall(
            r'(?:Dr|Pr|Interne|CCA)\s+([A-ZÉÈÊËÀÂÄÔÖÛÜÇ][a-zéèêëàâäôöûüç]+(?:\s+[A-ZÉÈÊËÀÂÄÔÖÛÜÇ]+)?)',
            anonymized
        ))
        for name in doctor_names:
            if not self.is_similar(name, lastname, firstname):
                anonymized = re.sub(re.escape(name), '[MEDICAL_STAFF]', anonymized, flags=re.IGNORECASE)
        
        # STEP 7: Remove remaining PII
        anonymized = re.sub(r'[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}', '[EMAIL_REMOVED]', 
                           anonymized, flags=re.IGNORECASE)
        anonymized = re.sub(r'\b0[1-9][\s.]?\d{2}[\s.]?\d{2}[\s.]?\d{2}[\s.]?\d{2}\b', 
                           '[PHONE_REMOVED]', anonymized)
        
        return anonymized, code, count
    
    def save_mapping(self, folder_name=None):
        """Save patient mapping and track processed folders."""
        if folder_name:
            self.processed_folders.add(folder_name)
        
        csv_path = self.output_dir / "patient_mapping.csv"
        with open(csv_path, 'w', encoding='utf-8', newline='') as f:
            f.write("Code,Lastname,Firstname,Full_Name,Updated\n")
            for name, info in sorted(self.patient_mapping.items(), key=lambda x: x[1]['code']):
                f.write(f"{info['code']},{info['lastname']},{info['firstname']},\"{name}\",{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        
        json_path = self.output_dir / "patient_mapping.json"
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump({
                'updated': datetime.now().isoformat(),
                'total_patients': len(self.patient_mapping),
                'processed_folders': list(self.processed_folders),
                'mappings': [
                    {'code': info['code'], 'lastname': info['lastname'], 'firstname': info['firstname']}
                    for _, info in sorted(self.patient_mapping.items(), key=lambda x: x[1]['code'])
                ]
            }, f, ensure_ascii=False, indent=2)

processor = MedicalProcessor()
print("✓ Processor ready")

✓ Processor ready


In [3]:
# CONFIGURE THIS
root_directory = r"C:\Users\Alina\Documents\stage_tony\data_sample"  # ← CHANGE THIS ONCE

root_path = Path(root_directory)

# Find all folders with PDFs
all_folders = []
for item in root_path.rglob("*"):
    if item.is_dir() and list(item.glob("*.pdf")):
        # Skip system folders
        if item.name.startswith('.') or item.name in ['__pycache__', '.ipynb_checkpoints']:
            continue
        all_folders.append(item)

# Filter out already processed
remaining_folders = [f for f in all_folders if f.name not in processor.processed_folders]

print(f"{'='*70}")
print(f"FOLDER ANALYSIS")
print(f"{'='*70}")
print(f"Total folders with PDFs: {len(all_folders)}")
print(f"Already processed: {len(all_folders) - len(remaining_folders)}")
print(f"Remaining to process: {len(remaining_folders)}")
print(f"{'='*70}\n")

if remaining_folders:
    print("Next folders to process:")
    for i, folder in enumerate(remaining_folders[:10], 1):
        print(f"  {i}. {folder.name}")
    if len(remaining_folders) > 10:
        print(f"  ... and {len(remaining_folders) - 10} more")
else:
    print("✅ All folders already processed!")

FOLDER ANALYSIS
Total folders with PDFs: 2
Already processed: 0
Remaining to process: 2

Next folders to process:
  1. 2006 RDB 0015
  2. 2006 RDB 0190


In [4]:
print(f"{'='*70}")
print(f"STARTING AUTOMATED PROCESSING")
print(f"{'='*70}\n")

for idx, folder_path in enumerate(remaining_folders, 1):
    try:
        print(f"\n[{idx}/{len(remaining_folders)}] Processing: {folder_path.name}")
        print("-" * 70)
        
        # Find PDFs
        pdfs = list(folder_path.glob("*.pdf"))
        print(f"  Found {len(pdfs)} PDF(s)")
        
        # Check cache first
        if folder_path.name in processor.folder_patient_cache:
            lastname, firstname = processor.folder_patient_cache[folder_path.name]
            print(f"  ✓ Using cached patient name: {lastname}, {firstname}")
        else:
            # OCR all PDFs
            combined_text = ""
            for pdf in pdfs:
                print(f"  OCR: {pdf.name}...", end=" ")
                text, method = processor.ocr_pdf(pdf)
                print(f"✓ ({method})")
                combined_text += f"\n\n{'='*50}\nFILE: {pdf.name}\n{'='*50}\n\n{text}"
            
            # Extract patient name
            print("  Extracting patient name...", end=" ")
            lastname, firstname, source = processor.extract_patient_name(combined_text)
            
            if not lastname:
                print("⚠ MANUAL INPUT NEEDED")
                print(f"\n  {'='*68}")
                print(f"  Folder: {folder_path.name}")
                print(f"  Location: {folder_path}")
                print(f"  PDF files in folder:")
                for pdf in pdfs[:5]:
                    print(f"    - {pdf.name}")
                if len(pdfs) > 5:
                    print(f"    ... and {len(pdfs) - 5} more")
                print(f"  {'='*68}")
                print("\n  Enter patient name (applies to ALL files in this folder):")
                lastname = input("    LASTNAME (multi-word OK, e.g., NDOUNKE DJATCHEU): ").strip().upper()
                firstname = input("    FIRSTNAME (multi-word OK, e.g., MARVIN BRYAN): ").strip().upper()
                print(f"  {'='*68}\n")
            else:
                print(f"✓ {lastname}, {firstname} (from {source})")
            
            # Cache for this folder
            processor.folder_patient_cache[folder_path.name] = (lastname, firstname)
        
        # Re-OCR if we used cache
        if 'combined_text' not in locals() or not combined_text:
            combined_text = ""
            for pdf in pdfs:
                print(f"  OCR: {pdf.name}...", end=" ")
                text, method = processor.ocr_pdf(pdf)
                print(f"✓ ({method})")
                combined_text += f"\n\n{'='*50}\nFILE: {pdf.name}\n{'='*50}\n\n{text}"
        
        # Anonymize
        print("  Anonymizing...", end=" ")
        anonymized, code, count = processor.anonymize(combined_text, lastname, firstname)
        print(f"✓ {code} ({count} replacements)")
        
        # Save
        output_dir = processor.output_dir / "anonymized_texts"
        output_dir.mkdir(exist_ok=True)
        output_file = output_dir / f"{code}_{folder_path.name}.txt"
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(anonymized)
        print(f"  Saved: {output_file.name}")
        
        # Update mapping
        processor.save_mapping(folder_path.name)
        
        print(f"  ✅ COMPLETE")
        
        # Reset combined_text for next iteration
        combined_text = ""
        
        # Small delay
        time.sleep(0.5)
    
    except KeyboardInterrupt:
        print("\n\n⚠️  Interrupted by user")
        print(f"Progress saved. {idx-1}/{len(remaining_folders)} folders completed.")
        break
    
    except Exception as e:
        print(f"\n  ❌ ERROR: {e}")
        print(f"  Skipping folder: {folder_path.name}")
        import traceback
        traceback.print_exc()
        print(f"  You can fix and rerun this cell to continue\n")

print(f"\n{'='*70}")
print(f"BATCH PROCESSING COMPLETE")
print(f"{'='*70}")
print(f"Total patients: {len(processor.patient_mapping)}")
print(f"Outputs: {processor.output_dir}")

STARTING AUTOMATED PROCESSING


[1/2] Processing: 2006 RDB 0015
----------------------------------------------------------------------
  Found 9 PDF(s)
  OCR: DOC_00177.pdf... ✓ (ocr)
  OCR: DOC_00178.pdf... ✓ (ocr)
  OCR: DOC_00179.pdf... ✓ (ocr)
  OCR: DOC_00180.pdf... ✓ (ocr)
  OCR: DOC_00181.pdf... ✓ (ocr)
  OCR: DOC_00182.pdf... ✓ (ocr)
  OCR: DOC_00183.pdf... ✓ (ocr)
  OCR: DOC_00184.pdf... ✓ (ocr)
  OCR: RDB 0015.pdf... ✓ (ocr)
  Extracting patient name... ✓ NDOUNKE, MARVIN (from Nom/Prénom)
  Anonymizing... ✓ PATIENT_001 (25 replacements)
  Saved: PATIENT_001_2006 RDB 0015.txt
  ✅ COMPLETE

[2/2] Processing: 2006 RDB 0190
----------------------------------------------------------------------
  Found 4 PDF(s)
  OCR: 2006 RDB 0190.pdf... ✓ (ocr)
  OCR: DOC_00150.pdf... ✓ (ocr)
  OCR: DOC_00151.pdf... ✓ (ocr)
  OCR: DOC_00152.pdf... ✓ (ocr)
  Extracting patient name... ✓ ALI, SAID (from Nom/Prénom)
  Anonymizing... ✓ PATIENT_002 (8 replacements)
  Saved: PATIENT_002_2006 RDB 0190.

In [5]:
# Debug: Check what OCR extracted
test_pdf = r"C:\Users\Alina\Documents\stage_tony\data_sample\2006 RDB 0015\DOC_00177.pdf"

doc = fitz.open(test_pdf)
page = doc[0]
pix = page.get_pixmap(dpi=300)
img_data = pix.tobytes("png")

from io import BytesIO
img = Image.open(BytesIO(img_data))
text = pytesseract.image_to_string(img, lang='fra', config='--psm 1')

# Show first 1000 chars
print(text[:1000])

# Search for patient name
import re
pattern = r"de l'enfant\s+([A-ZÉÈÊËÀÂÄÔÖÛÜÇ][A-ZÉÈÊËÀÂÄÔÖÛÜÇ\s-]+),\s*([A-ZÉÈÊËÀÂÄÔÖÛÜÇ][A-ZÉÈÊËÀÂÄÔÖÛÜÇ\s-]+)"
match = re.search(pattern, text)
if match:
    print(f"\nFound: '{match.group(1)}' , '{match.group(2)}'")
else:
    print("\nPattern not found in OCR text")

ASSISTANCE HÔPITAUX
PUBLIQUE DE PARIS

Hôpital Robert Debré

48 boulevard Sérurier 75935 Paris Cedex 19

Code FINESS 75803454

PÔLE DE PÉDIATRIE AIGUË

& MÉDECINE INTERNE

PÉDIATRIE GÉNÉRALE

POINT MERT — 4{"° étage

Chef de Service .
Pr Antoine BOURRILLON

antämgwﬁl&gﬁ@.ænhgfr

Equipe médicale

Dr Aibert FAYE, PH
albemfaze@db.ag*he.fr

Dr François ANGOULVANT, CCA
francois.angoulwÆ@db.aghg.ﬁ
Dr Laure DE LOS ANGELES, CCA
lauæ.delosangglæ@ﬂb.aghgfr
Dr Véronique HOUDOUIN, CCA
vemnigue.houdguia@db.aghg.ﬁ

Dr Mathie LORROT, CCA
h

mathie.lorrot@rdb:aphp.fr

Tél : 01.40.03.20.48
Fax : 01.40.03.20.43

Imprimé le : 21/04/2013

COMPTE RENDU DE SEJOUR EN PEDIATRIE GENERALE
DU 29/08/06 AU 01/09/06

de l’enfant NDOUNKE DJATCHEU, MARVIN BRYAN
Né(e) le 05/11/2000 (M)

NPI :006030170022 N°de séjour : 7006236547
Adresse des parents :

10 ALLEE MICKAEL LEFEBVRE
92600 ASNIERES SUR SEINE

(Compte rendu Validé)

Destinataires

- Equipe Medicale PMI - 4 route d'Asnieres Sur Seine - 92600 Asnieres Sur Seine